# Throughput Analysis: Baseline vs DRO

Paper-ready visualizations for GPU utilization comparison.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Paper style
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.figsize': (8, 5),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

# Load data
with open('../../logs/analysis_log/throughput_comparison.json', 'r') as f:
    data = json.load(f)

baseline = data['baseline']
dro = data['dro']

# Skip warmup
skip = 10
baseline_tflops = np.array(baseline['tflops'][skip:])
dro_tflops = np.array(dro['tflops'][skip:])
baseline_time = np.array(baseline['elapsed_ms'][skip:])
dro_time = np.array(dro['elapsed_ms'][skip:])

## Figure 1: Throughput Distribution (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

box_data = [baseline_tflops, dro_tflops]
positions = [1, 2]
colors = ['#2E86AB', '#A23B72']

bp = ax.boxplot(box_data, positions=positions, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(['Baseline', 'DRO (Ours)'])
ax.set_ylabel('Throughput (TFLOP/s/GPU)')
ax.set_title('Training Throughput Comparison')
ax.grid(axis='y', alpha=0.3)

# Add mean markers
means = [np.mean(baseline_tflops), np.mean(dro_tflops)]
ax.scatter(positions, means, marker='D', color='white', edgecolor='black', s=50, zorder=5, label='Mean')

# Add annotation
diff_pct = (means[1] - means[0]) / means[0] * 100
ax.annotate(f'+{diff_pct:.1f}%', xy=(2.3, means[1]), fontsize=12, fontweight='bold', color='#A23B72')

plt.tight_layout()
plt.savefig('throughput_boxplot.pdf')
plt.savefig('throughput_boxplot.png')
plt.show()

## Figure 2: Throughput Over Training (Time Series)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Smooth with rolling average
window = 20
baseline_smooth = np.convolve(baseline_tflops, np.ones(window)/window, mode='valid')
dro_smooth = np.convolve(dro_tflops, np.ones(window)/window, mode='valid')

iterations = np.array(baseline['iterations'][skip:])
iter_smooth = iterations[:len(baseline_smooth)]

ax.plot(iter_smooth, baseline_smooth, label='Baseline', color='#2E86AB', linewidth=1.5)
ax.plot(iter_smooth, dro_smooth, label='DRO (Ours)', color='#A23B72', linewidth=1.5)

ax.set_xlabel('Training Iteration')
ax.set_ylabel('Throughput (TFLOP/s/GPU)')
ax.set_title('Training Throughput Over Time (20-iteration moving average)')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

# Add horizontal lines for means
ax.axhline(np.mean(baseline_tflops), color='#2E86AB', linestyle='--', alpha=0.5)
ax.axhline(np.mean(dro_tflops), color='#A23B72', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('throughput_timeseries.pdf')
plt.savefig('throughput_timeseries.png')
plt.show()

## Figure 3: Iteration Time Distribution (Histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bins = np.linspace(5800, 7200, 50)

ax.hist(baseline_time, bins=bins, alpha=0.6, label='Baseline', color='#2E86AB', edgecolor='black', linewidth=0.5)
ax.hist(dro_time, bins=bins, alpha=0.6, label='DRO (Ours)', color='#A23B72', edgecolor='black', linewidth=0.5)

# Add vertical lines for means
ax.axvline(np.mean(baseline_time), color='#2E86AB', linestyle='--', linewidth=2, label=f'Baseline mean: {np.mean(baseline_time):.0f}ms')
ax.axvline(np.mean(dro_time), color='#A23B72', linestyle='--', linewidth=2, label=f'DRO mean: {np.mean(dro_time):.0f}ms')

ax.set_xlabel('Time per Iteration (ms)')
ax.set_ylabel('Frequency')
ax.set_title('Iteration Time Distribution')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('iteration_time_histogram.pdf')
plt.savefig('iteration_time_histogram.png')
plt.show()

## Figure 4: Summary Bar Chart with Confidence Intervals

In [ ]:
def bootstrap_ci(data, n_bootstrap=10000, ci=95):
    """Compute bootstrap confidence interval."""
    means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        means.append(np.mean(sample))
    lower = np.percentile(means, (100-ci)/2)
    upper = np.percentile(means, 100-(100-ci)/2)
    return lower, upper

# Compute CIs
baseline_ci = bootstrap_ci(baseline_tflops)
dro_ci = bootstrap_ci(dro_tflops)

fig, ax = plt.subplots(figsize=(6, 5))

methods = ['Baseline', 'DRO (Ours)']
means = [np.mean(baseline_tflops), np.mean(dro_tflops)]
ci_lower = [means[0] - baseline_ci[0], means[1] - dro_ci[0]]
ci_upper = [baseline_ci[1] - means[0], dro_ci[1] - means[1]]

colors = ['#2E86AB', '#A23B72']
bars = ax.bar(methods, means, yerr=[ci_lower, ci_upper], capsize=8, color=colors, alpha=0.8, edgecolor='black')

ax.set_ylabel('Throughput (TFLOP/s/GPU)')
ax.set_title('Mean Training Throughput (95% CI)')
ax.set_ylim(60, 70)

# Add value labels
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean + 0.8, f'{mean:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('throughput_barplot.pdf')
plt.savefig('throughput_barplot.png')
plt.show()

print(f"Baseline: {means[0]:.2f} TFLOP/s/GPU (95% CI: [{baseline_ci[0]:.2f}, {baseline_ci[1]:.2f}])")
print(f"DRO:      {means[1]:.2f} TFLOP/s/GPU (95% CI: [{dro_ci[0]:.2f}, {dro_ci[1]:.2f}])")

## Statistical Summary Table

In [ ]:
import pandas as pd

# Create summary table
summary = pd.DataFrame({
    'Metric': [
        'Mean TFLOP/s/GPU',
        'Std TFLOP/s/GPU', 
        'CV (%)',
        'Mean Time (ms)',
        'Std Time (ms)',
        'P10 Time (ms)',
        'P90 Time (ms)'
    ],
    'Baseline': [
        f"{np.mean(baseline_tflops):.2f}",
        f"{np.std(baseline_tflops):.2f}",
        f"{np.std(baseline_tflops)/np.mean(baseline_tflops)*100:.2f}",
        f"{np.mean(baseline_time):.1f}",
        f"{np.std(baseline_time):.1f}",
        f"{np.percentile(baseline_time, 10):.1f}",
        f"{np.percentile(baseline_time, 90):.1f}"
    ],
    'DRO': [
        f"{np.mean(dro_tflops):.2f}",
        f"{np.std(dro_tflops):.2f}",
        f"{np.std(dro_tflops)/np.mean(dro_tflops)*100:.2f}",
        f"{np.mean(dro_time):.1f}",
        f"{np.std(dro_time):.1f}",
        f"{np.percentile(dro_time, 10):.1f}",
        f"{np.percentile(dro_time, 90):.1f}"
    ]
})

# Calculate difference
summary['Δ (%)'] = [
    f"+{(np.mean(dro_tflops) - np.mean(baseline_tflops))/np.mean(baseline_tflops)*100:.1f}",
    f"{(np.std(dro_tflops) - np.std(baseline_tflops))/np.std(baseline_tflops)*100:+.1f}",
    '-',
    f"{(np.mean(dro_time) - np.mean(baseline_time))/np.mean(baseline_time)*100:+.1f}",
    f"{(np.std(dro_time) - np.std(baseline_time))/np.std(baseline_time)*100:+.1f}",
    '-',
    '-'
]

print(summary.to_markdown(index=False))

# Export as LaTeX
latex_table = summary.to_latex(index=False, escape=False)
print("\n\nLaTeX table:")
print(latex_table)

## Statistical Significance Test

In [ ]:
# Welch's t-test (doesn't assume equal variance)
t_stat, p_value = stats.ttest_ind(baseline_tflops, dro_tflops, equal_var=False)

# Effect size (Cohen's d)
pooled_std = np.sqrt((np.var(baseline_tflops) + np.var(dro_tflops)) / 2)
cohens_d = (np.mean(dro_tflops) - np.mean(baseline_tflops)) / pooled_std

print("=== Statistical Significance ===")
print(f"Welch's t-test: t = {t_stat:.4f}, p = {p_value:.2e}")
print(f"Cohen's d (effect size): {cohens_d:.4f}")
print(f"\nInterpretation:")
print(f"  - The difference is {'statistically significant' if p_value < 0.05 else 'not significant'} (p < 0.05)")
if abs(cohens_d) < 0.2:
    effect = 'negligible'
elif abs(cohens_d) < 0.5:
    effect = 'small'
elif abs(cohens_d) < 0.8:
    effect = 'medium'
else:
    effect = 'large'
print(f"  - Effect size is {effect} (|d| = {abs(cohens_d):.2f})")